In [1]:
!pip install lime
!pip install kagglehub transformers torch -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 4.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283834 sha256=614835f53ce6fc08f370737db1e1bdd34afdad7647285d8455175aa66fc82ea7
  Stored in directory: /root/.cache/pip/wheels/e7/5d/0e/4b4fff9a47468fed5633211fb3b76d1db43fe806a17fb7486a
Successfully built lime


In [2]:
import pandas as pd
import numpy as np
import pickle
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub


In [3]:
path = kagglehub.dataset_download("shivamb/real-or-fake-fake-jobposting-prediction")
df = pd.read_csv(f"{path}/fake_job_postings.csv")

print(f"✅ Dataset loaded: {df.shape}")
print(f"   Fraud rate: {df['fraudulent'].mean()*100:.2f}%")


100%|██████████| 16.1M/16.1M [00:00<00:00, 57.5MB/s]

Extracting files...


✅ Dataset loaded: (17880, 18)
   Fraud rate: 4.84%


In [4]:
# Text preprocessing
import nltk
nltk.download('punkt_tab')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if pd.isna(text):
        return ""
    tokens = word_tokenize(str(text).lower())
    tokens = [token for token in tokens if token.isalpha() and token not in stop_words]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return ' '.join(tokens)

# Combine text fields (same as your training)
def combine_text_fields(row):
    fields = ['title', 'location', 'company_profile', 'description',
              'requirements', 'benefits', 'required_experience',
              'required_education', 'industry', 'function']
    text_parts = []
    for field in fields:
        if pd.notna(row.get(field)):
            text_parts.append(str(row[field]))
    return ' '.join(text_parts) if text_parts else "unknown job"

print("Combining text fields...")
df['combined_text'] = df.apply(combine_text_fields, axis=1)

print("Preprocessing text...")
df['text_processed'] = df['combined_text'].apply(preprocess_text)

# Features
df['location_fraud_ratio'] = df.groupby('location')['fraudulent'].transform('mean').fillna(0.05)
df['character_count'] = df['combined_text'].str.len()

# Create feature set
X = df[['text_processed', 'telecommuting', 'has_company_logo',
         'has_questions', 'location_fraud_ratio', 'character_count']]
y = df['fraudulent'].values

# Train/test split (same random state as your training!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"✅ Data prepared")
print(f"   Train: {X_train.shape[0]} samples ({y_train.mean()*100:.2f}% fraud)")
print(f"   Test:  {X_test.shape[0]} samples ({y_test.mean()*100:.2f}% fraud)")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Combining text fields...
Preprocessing text...
✅ Data prepared
   Train: 12516 samples (4.84% fraud)
   Test:  5364 samples (4.85% fraud)


In [5]:
# Remove existing directory if it exists
import shutil
import os

if os.path.exists('job-postings-fraud'):
    shutil.rmtree('job-postings-fraud')
    print("🗑️ Removed existing directory")

!git clone https://github.com/tommygarner/job-postings-fraud.git
models_path = "job-postings-fraud/models/"
print(f"Cloned repo, models at: {models_path}")

Cloning into 'job-postings-fraud'...
remote: Enumerating objects: 270, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 270 (delta 4), reused 8 (delta 3), pack-reused 261 (from 1)
Receiving objects: 100% (270/270), 11.47 MiB | 10.25 MiB/s, done.
Resolving deltas: 100% (105/105), done.
Cloned repo, models at: job-postings-fraud/models/


In [24]:
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
)
from tensorflow.keras.models import load_model
from pathlib import Path
import pickle
import torch


print("\n" + "="*80)
print("LOADING MODELS FROM GITHUB...")
print("="*80)

models_path = "job-postings-fraud/models/"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    # Load individual NB components (pipeline is corrupted)
    print("⚠️ Note: Using individual NB components (nb_pipeline.pkl has issues)")

    with open(f"{models_path}naive_bayes_model.pkl", "rb") as f:
        nb_model = pickle.load(f)
        print("   ✅ Loaded naive_bayes_model.pkl")

    with open(f"{models_path}vectorizer.pkl", "rb") as f:
        vectorizer = pickle.load(f)
        print("   ✅ Loaded vectorizer.pkl")

    # Load LSTM tokenizer
    with open(f"{models_path}tokenizer.pkl", "rb") as f:
        lstm_tokenizer = pickle.load(f)
        print("   ✅ Loaded tokenizer.pkl")

    # Load LSTM model
    lstm_model = load_model(f"{models_path}lstm_model.h5")
    print("   ✅ Loaded lstm_model.h5")

    # Load MiniLM (in subfolder)
    minilm_dir = f"{models_path}model_miniLM_final/"

    # Load config explicitly
    minilm_config = AutoConfig.from_pretrained(minilm_dir)

    # Force sequence-classification head using that config
    minilm_tokenizer = AutoTokenizer.from_pretrained(minilm_dir)
    minilm_model = AutoModelForSequenceClassification.from_pretrained(
        minilm_dir,
        config=minilm_config
    )
    minilm_model.to(device)
    minilm_model.eval()
    print("   ✅ Loaded model_miniLM_final/")

    print("\n✅ All models loaded successfully!")

except FileNotFoundError as e:
    print(f"❌ Error: File not found - {e}")
    print(f"\nChecking what's in {models_path}:")
    !ls -lh {models_path}

except Exception as e:
    print(f"❌ Error loading models: {e}")
    import traceback
    traceback.print_exc()



LOADING MODELS FROM GITHUB...
⚠️ Note: Using individual NB components (nb_pipeline.pkl has issues)
   ✅ Loaded naive_bayes_model.pkl
   ✅ Loaded vectorizer.pkl


   ✅ Loaded tokenizer.pkl
   ✅ Loaded lstm_model.h5
   ✅ Loaded model_miniLM_final/

✅ All models loaded successfully!


In [25]:
from scipy.sparse import hstack, csr_matrix

def extract_nb_features(df_subset, vectorizer):
    text_features = vectorizer.transform(df_subset['text_processed'])
    numeric = df_subset[['telecommuting',
                         'has_company_logo',
                         'has_questions',
                         'location_fraud_ratio',
                         'character_count']].fillna(0).values
    numeric_sparse = csr_matrix(numeric)
    return hstack([text_features, numeric_sparse])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
minilm_model.to(device)

# Precompute test texts once
test_texts = df.loc[X_test.index, "combined_text"].tolist()


In [26]:
!pip install -q tqdm

In [13]:
from tqdm.notebook import tqdm  # not tqdm.auto

print("\n" + "="*80)
print("GENERATING PREDICTIONS ON TEST SET...")
print("="*80)

# 1. NB
X_test_nb = extract_nb_features(X_test, vectorizer)
nb_probs = nb_model.predict_proba(X_test_nb)[:, 1]

# 2. LSTM
X_test_lstm = pad_sequences(
    lstm_tokenizer.texts_to_sequences(X_test["text_processed"].tolist()),
    maxlen=200
)
lstm_probs = lstm_model.predict(X_test_lstm, verbose=0).ravel()

# 3. MiniLM with visible progress bar
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
minilm_model.to(device)

test_texts = df.loc[X_test.index, "combined_text"].tolist()
minilm_probs = []

batch_size = 128
max_len = 128

for i in tqdm(range(0, len(test_texts), batch_size), desc="MiniLM batches"):
    batch_texts = test_texts[i:i + batch_size]
    inputs = minilm_tokenizer(
        batch_texts,
        return_tensors="pt",
        truncation=True,
        max_length=max_len,
        padding=True
    ).to(device)

    with torch.no_grad():
        logits = minilm_model(**inputs).logits
        probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

    minilm_probs.extend(probs)

minilm_probs = np.array(minilm_probs)

print("NB mean prob:", nb_probs.mean())
print("LSTM mean prob:", lstm_probs.mean())
print("MiniLM mean prob:", minilm_probs.mean())



GENERATING PREDICTIONS ON TEST SET...


MiniLM batches:   0%|          | 0/42 [00:00<?, ?it/s]

NB mean prob: 0.03563280584776362
LSTM mean prob: 0.056169305
MiniLM mean prob: 0.040030114


In [14]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
import numpy as np
import pandas as pd

def evaluate_model(y_true, probs, threshold=0.5, name="Model"):
    preds = (probs >= threshold).astype(int)
    return {
        "model": name,
        "accuracy": accuracy_score(y_true, preds),
        "f1":        f1_score(y_true, preds),
        "precision": precision_score(y_true, preds),
        "recall":    recall_score(y_true, preds),
        "roc_auc":   roc_auc_score(y_true, probs),
    }

print("Individual models:")
for name, probs in [
    ("Naive Bayes", nb_probs),
    ("LSTM",        lstm_probs),
    ("MiniLM+IG",   minilm_probs),
]:
    m = evaluate_model(y_test, probs, name=name)
    print(f"{name:11} | Acc {m['accuracy']:.4f}  F1 {m['f1']:.4f}  "
          f"Prec {m['precision']:.4f}  Rec {m['recall']:.4f}  AUC {m['roc_auc']:.4f}")

# ---------- Ensemble search ----------

def eval_ensemble(y_true, nb_p, lstm_p, mini_p, w_nb, w_lstm, w_minilm, thr=0.5):
    total = w_nb + w_lstm + w_minilm
    if total == 0:
        return None
    w_nb, w_lstm, w_minilm = w_nb/total, w_lstm/total, w_minilm/total
    ens_probs = w_nb*nb_p + w_lstm*lstm_p + w_minilm*mini_p
    preds = (ens_probs >= thr).astype(int)
    return {
        "w_nb": round(w_nb, 3),
        "w_lstm": round(w_lstm, 3),
        "w_minilm": round(w_minilm, 3),
        "accuracy": accuracy_score(y_true, preds),
        "f1":        f1_score(y_true, preds),
        "precision": precision_score(y_true, preds),
        "recall":    recall_score(y_true, preds),
        "roc_auc":   roc_auc_score(y_true, ens_probs),
    }

results = []

# Some hand-picked combos
base_combos = [
    (1, 0, 0),   # NB only
    (0, 1, 0),   # LSTM only
    (0, 0, 1),   # MiniLM only
    (1, 1, 1),   # equal
    (0.5, 0.5, 0),
    (0.2, 0.3, 0.5),
    (0.2, 0.25, 0.55),
    (0.15, 0.15, 0.7),
    (0.1, 0.1, 0.8),
    (0.05, 0.05, 0.9),
]

# Grid around MiniLM-heavy weights
for w_minilm in np.arange(0.5, 1.01, 0.05):
    for w_nb in np.arange(0, 1.01 - w_minilm, 0.05):
        w_lstm = 1 - w_minilm - w_nb
        if w_lstm < 0:
            continue
        base_combos.append((w_nb, w_lstm, w_minilm))

for w_nb, w_lstm, w_minilm in base_combos:
    r = eval_ensemble(y_test, nb_probs, lstm_probs, minilm_probs,
                      w_nb, w_lstm, w_minilm)
    if r:
        results.append(r)

results_df = pd.DataFrame(results).drop_duplicates(
    subset=["w_nb", "w_lstm", "w_minilm"]
)
results_df = results_df.sort_values("f1", ascending=False)

print("\nTop 10 ensembles by F1:")
print(results_df.head(10).to_string(index=False))

best = results_df.iloc[0]
print("\nBest ensemble weights:")
print(f"NB={best['w_nb']:.3f}, LSTM={best['w_lstm']:.3f}, MiniLM={best['w_minilm']:.3f}")
print(f"Accuracy={best['accuracy']:.4f}, F1={best['f1']:.4f}, "
      f"Precision={best['precision']:.4f}, Recall={best['recall']:.4f}, "
      f"AUC={best['roc_auc']:.4f}")


Individual models:
Naive Bayes | Acc 0.9702  F1 0.5960  Prec 0.8676  Rec 0.4538  AUC 0.8494
LSTM        | Acc 0.9737  F1 0.6994  Prec 0.7847  Rec 0.6308  AUC 0.9343
MiniLM+IG   | Acc 0.9700  F1 0.6508  Prec 0.7463  Rec 0.5769  AUC 0.8921

Top 10 ensembles by F1:
 w_nb  w_lstm  w_minilm  accuracy       f1  precision   recall  roc_auc
0.333   0.333     0.333  0.978188 0.709677   1.000000 0.550000 0.973900
0.000   1.000     0.000  0.973714 0.699360   0.784689 0.630769 0.934257
0.450   0.050     0.500  0.975019 0.688372   0.870588 0.569231 0.966882
0.400   0.100     0.500  0.975019 0.688372   0.870588 0.569231 0.970661
0.350   0.150     0.500  0.975019 0.688372   0.870588 0.569231 0.972096
0.250   0.250     0.500  0.974832 0.688222   0.861272 0.573077 0.972871
0.050   0.450     0.500  0.974646 0.688073   0.852273 0.576923 0.969424
0.200   0.300     0.500  0.974646 0.685185   0.860465 0.569231 0.972459
0.300   0.200     0.500  0.974646 0.685185   0.860465 0.569231 0.972726
0.150   0.350    

In [16]:
# Ensemble with best weights
w_nb, w_lstm, w_minilm = 0.33, 0.33, 0.33
total = w_nb + w_lstm + w_minilm
w_nb, w_lstm, w_minilm = w_nb/total, w_lstm/total, w_minilm/total

ensemble_probs = w_nb*nb_probs + w_lstm*lstm_probs + w_minilm*minilm_probs
ensemble_preds = (ensemble_probs >= 0.5).astype(int)

print("Total test samples:", len(y_test))
print("True fraud count:  ", y_test.sum())
print("Pred fraud count:  ", ensemble_preds.sum())
print("Pred legit count:  ", (ensemble_preds == 0).sum())


Total test samples: 5364
True fraud count:   260
Pred fraud count:   143
Pred legit count:   5221


In [17]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, ensemble_preds)
tn, fp, fn, tp = cm.ravel()
print("Confusion matrix:\n", cm)
print(f"TN={tn}, FP={fp}, FN={fn}, TP={tp}")


Confusion matrix:
 [[5104    0]
 [ 117  143]]
TN=5104, FP=0, FN=117, TP=143


In [18]:
fraud_indices = np.where(y_test == 1)[0][:10]
print("Ensemble probs for first 10 fraud cases:")
print(ensemble_probs[fraud_indices])
print("Predictions:", ensemble_preds[fraud_indices])


Ensemble probs for first 10 fraud cases:
[0.51813992 0.02070613 0.63955131 0.97991658 0.46300785 0.61629315
 0.34461619 0.1424071  0.97549868 0.95056258]
Predictions: [1 0 1 1 0 1 0 0 1 1]


In [22]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

w_nb, w_lstm, w_minilm = 0.33, 0.33, 0.33
total = w_nb + w_lstm + w_minilm
w_nb, w_lstm, w_minilm = w_nb/total, w_lstm/total, w_minilm/total

ensemble_probs = w_nb*nb_probs + w_lstm*lstm_probs + w_minilm*minilm_probs

thresholds = np.linspace(0.15, 0.8, 14)  # e.g., 0.30, 0.35, ..., 0.80
rows = []

for thr in thresholds:
    preds = (ensemble_probs >= thr).astype(int)
    rows.append({
        "threshold": thr,
        "accuracy":  accuracy_score(y_test, preds),
        "f1":        f1_score(y_test, preds),
        "precision": precision_score(y_test, preds),
        "recall":    recall_score(y_test, preds),
        "roc_auc":   roc_auc_score(y_test, ensemble_probs),  # same across thresholds
    })

thr_df = pd.DataFrame(rows).sort_values("recall", ascending=False)
print(thr_df.to_string(index=False))


 threshold  accuracy       f1  precision   recall  roc_auc
      0.15  0.964206 0.698113   0.590426 0.853846   0.9739
      0.20  0.972222 0.742660   0.673981 0.826923   0.9739
      0.25  0.975951 0.763303   0.729825 0.800000   0.9739
      0.30  0.978934 0.782274   0.783784 0.780769   0.9739
      0.35  0.980052 0.763797   0.896373 0.665385   0.9739
      0.40  0.978934 0.732861   0.950920 0.596154   0.9739
      0.45  0.979866 0.737864   1.000000 0.584615   0.9739
      0.50  0.978188 0.709677   1.000000 0.550000   0.9739
      0.55  0.975951 0.670077   1.000000 0.503846   0.9739
      0.60  0.974273 0.638743   1.000000 0.469231   0.9739
      0.65  0.970544 0.563536   1.000000 0.392308   0.9739
      0.70  0.967748 0.501441   1.000000 0.334615   0.9739
      0.75  0.965884 0.456973   1.000000 0.296154   0.9739
      0.80  0.964392 0.419453   1.000000 0.265385   0.9739
